## Import libraries


In [13]:
import os
import re
from glob import glob

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio as rio
import rioxarray
import xarray as xr
from rasterio.windows import Window
from shapely.geometry import Point
from tqdm.auto import tqdm

# --- Repo-relative paths ---------------------------------------------------
# Resolved from this notebook's location, so the checkout can be moved or cloned
# anywhere without editing paths here. Override with SOIL_SCENARIOS_ROOT.
import os
from pathlib import Path


def _find_repo_root(start=None):
    env = os.environ.get("SOIL_SCENARIOS_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path(start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "simplace").is_dir() and (cand / "orchestration").is_dir():
            return cand
    raise FileNotFoundError(
        f"repo root not found from {here} (looked for simplace/ + orchestration/)"
    )


REPO_ROOT = _find_repo_root()

# External (outside the repo): shared DE crop-type rasters on the cluster store.
MAIN_DATA_DIR = os.environ.get("EXTERNAL_DATA_DIR", "/beegfs/halder/DATA")
INTERIM_DATA_DIR = os.path.join(str(REPO_ROOT), "data", "interim")
PROCESSED_DATA_DIR = os.path.join(str(REPO_ROOT), "data", "processed")

## Function to sample the raster efficiently with batching


In [14]:
def sample_raster(raster_path, output_path, samples_per_class=500, batch_rows=1000):

    candidates = []

    with rio.open(raster_path) as src:
        width = src.width
        height = src.height
        transform = src.transform
        crs = src.crs
        nodata = src.nodata if src.nodata is not None else -9999

        print(f"Processing raster: {width}x{height}")
        print(f"Batch size: {batch_rows} rows per read")

        # Loop through the image in vertical chunks (Batches)
        for row_offset in tqdm(range(0, height, batch_rows)):
            # Calculate the actual height of this batch (last batch might be smaller)
            current_batch_height = min(batch_rows, height - row_offset)

            # Define the window: (col_off, row_off, width, height)
            window = Window(0, row_offset, width, current_batch_height)

            # Read data
            data = src.read(1, window=window)

            # Create a mask for valid pixels (not 0 and not NoData)
            is_valid = (data != 0) & (data != nodata)

            if not np.any(is_valid):
                continue

            # Get coordinates relative to the batch
            valid_rows, valid_cols = np.where(is_valid)
            valid_values = data[valid_rows, valid_cols]

            # Convert to Global Coordinates immediately
            global_rows = valid_rows + row_offset
            global_cols = valid_cols

            # 3. Create a temporary DataFrame for this batch
            df_batch = pd.DataFrame(
                {"crop_id": valid_values, "row": global_rows, "col": global_cols}
            )

            # Greedy Sampling (Pre-filtering)
            sampled_batch = (
                df_batch.groupby("crop_id")
                .apply(
                    lambda x: x.sample(n=min(len(x), 100), random_state=42),
                    include_groups=False,
                )
                .reset_index(drop=False)
            )
            sampled_batch = sampled_batch[["crop_id", "row", "col"]]
            candidates.append(sampled_batch)

            print(
                f"Processed rows {row_offset} to {row_offset + current_batch_height}..."
            )

    # Post-Processing
    print("Aggregating candidates...")
    if not candidates:
        print("No valid pixels found!")
        return

    all_candidates = pd.concat(candidates, ignore_index=True)

    # Final Stratified Sampling
    final_df = (
        all_candidates.groupby("crop_id")
        .apply(
            lambda x: x.sample(n=min(len(x), samples_per_class), random_state=42),
            include_groups=False,
        )
        .reset_index(drop=False)
    )
    final_df = final_df[["crop_id", "row", "col"]]

    print("Converting to Lat/Lon...")
    # Convert row/col to real map coordinates
    with rio.open(raster_path) as src:
        xs, ys = rio.transform.xy(src.transform, final_df["row"], final_df["col"])

    geometry = [Point(x, y) for x, y in zip(xs, ys)]
    gdf = gpd.GeoDataFrame(final_df, geometry=geometry, crs=crs)

    gdf.to_file(output_path, driver="GPKG")
    print(f"Done! Saved {len(gdf)} samples to {output_path}")

    return gdf

In [15]:
# Load the raster paths
raster_paths = glob(
    os.path.join(MAIN_DATA_DIR, "DE_Crop_Types_2017_2021", "DE_Crop_Type_*.tif")
)

# for path in tqdm(raster_paths):
#     year = os.path.basename(path).replace(".tif", "").split("_")[-1]
#     print("*" * 20 + year + "*" * 20)
#     out_file_path = os.path.join(
#         INTERIM_DATA_DIR, "crop_type_samples_2017_2021", f"{year}_crop_type.gpkg"
#     )
#     result = sample_raster(
#         raster_path=path,
#         output_path=out_file_path,
#         samples_per_class=500,
#         batch_rows=10000,
#     )

## Prepare the final crop type sample points


In [16]:
# Read the shapefile of Germany
de_nuts1_gdf = gpd.read_file(
    os.path.join("/beegfs", "halder", "DATA", "DE_NUTS", "DE_NUTS_3.shp")
)
de_nuts1_gdf = de_nuts1_gdf[
    de_nuts1_gdf["LEVL_CODE"] == 1
]  # filter only NUTS1 level code
de_nuts1_gdf = de_nuts1_gdf[["NUTS_NAME", "geometry"]]
de_nuts1_gdf.rename(columns={"NUTS_NAME": "STATE_NAME"}, inplace=True)

print(de_nuts1_gdf.shape)
de_nuts1_gdf.head()

(16, 2)


,STATE_NAME,geometry
438,Schleswig-Holstein,"MULTIPOLYGON (((1254572.196 7257993.294, 12558..."
439,Mecklenburg-Vorpommern,"MULTIPOLYGON (((1424976.58 7254147.48, 1430472..."
440,Thüringen,"POLYGON ((1200657.108 6735655.986, 1202794.734..."
441,Niedersachsen,"MULTIPOLYGON (((966692.233 7148395.64, 967523...."
442,Baden-Württemberg,"MULTIPOLYGON (((1074230.536 6408356.046, 10738..."


In [17]:
crop_type_file_paths = glob(
    os.path.join(INTERIM_DATA_DIR, "crop_type_samples_2017_2021", "*.gpkg")
)

# Merge all the files
merged_gdf = pd.DataFrame()

for path in crop_type_file_paths:
    year = int(os.path.basename(path).replace(".gpkg", "").split("_")[0])
    gdf = gpd.read_file(path)
    gdf["year"] = year
    merged_gdf = pd.concat((merged_gdf, gdf), axis=0, ignore_index=True)

code_to_crop_type = {
    1110: "Wheat",
    1120: "Barley",
    1130: "Maize",
    1140: "Rice",
    1150: "Other cereals",
    1210: "Fresh Vegetables",
    1220: "Dry pulses",
    1310: "Potatoes",
    1320: "Sugar Beet",
    1410: "Sunflower",
    1420: "Soybeans",
    1430: "Rapeseed",
    1440: "Flax, cotton and hemp",
    2100: "Grapes",
    2200: "Olives",
    2310: "Fruits",
    2320: "Nuts",
    3100: "Unclassified arable crop",
    3200: "Unclassified permanent crop",
}

# Specify the crops under observation
crops = [
    "Wheat",
    "Barley",
    "Maize",
    "Sugar Beet",
    "Rapeseed",
    "Potatoes",
]  # Grain Maize, Spring Barley

merged_gdf = merged_gdf[merged_gdf["crop_id"] != 65535]  # drop the nodata samples
merged_gdf["crop_type"] = merged_gdf["crop_id"].replace(code_to_crop_type)
merged_gdf = merged_gdf.sort_values(by=["year", "crop_id"]).reset_index(drop=True)
merged_gdf.to_crs(crs=de_nuts1_gdf.crs, inplace=True)
merged_gdf = gpd.sjoin(
    left_df=de_nuts1_gdf, right_df=merged_gdf, how="right", predicate="intersects"
)
merged_gdf.drop(columns="index_left", inplace=True)
merged_gdf.dropna(inplace=True)
merged_gdf.to_crs(crs="EPSG:4326", inplace=True)
merged_gdf = merged_gdf[merged_gdf["crop_type"].isin(crops)]
merged_gdf["point_id"] = np.arange(len(merged_gdf))
merged_gdf = merged_gdf[["point_id", "year", "crop_id", "crop_type", "geometry"]]

# merged_gdf.to_file(os.path.join(PROCESSED_DATA_DIR, 'crop_type_samples', 'crop_type_samples.gpkg'))
merged_gdf = gpd.read_file(
    os.path.join(PROCESSED_DATA_DIR, "crop_type_samples", "crop_type_samples.gpkg")
)
print(merged_gdf.shape)
merged_gdf.head()

(8848, 5)


,point_id,year,crop_id,crop_type,geometry
0,0,2017,1110,Wheat,POINT (11.63824 49.10327)
1,1,2017,1110,Wheat,POINT (10.24612 48.47405)
2,2,2017,1110,Wheat,POINT (8.91542 51.21187)
3,3,2017,1110,Wheat,POINT (12.1151 50.04173)
4,4,2017,1110,Wheat,POINT (9.67718 50.80792)


## Download GLASS LAI data from 2017-2021

Website link: https://www.glass.hku.hk/download.html

Run this command in the terminal to download the data:

```bash
wget -r -np -nH --cut-dirs=4 -R "index.html*" -A "*h18v03*.hdf,*h18v04*.hdf,*h19v03*.hdf,*h19v04*.hdf" https://www.glass.hku.hk/archive/LAI/MODIS/250M/{2017..2021}/
```


## Extract Timeseries from GLASS LAI data


In [18]:
GLASS_LAI_ROOT_DIR = os.path.join(str(REPO_ROOT), "data", "raw", "GLASS_LAI")

SAMPLE_POINTS_PATH = os.path.join(
    PROCESSED_DATA_DIR, "crop_type_samples", "crop_type_samples.gpkg"
)

MODIS_SINU_CRS = (
    "+proj=sinu +lon_0=0 +x_0=0 +y_0=0 +a=6371007.181 +b=6371007.181 +units=m +no_defs"
)


def get_tile_id(filename):
    """Extracts tile ID (e.g., h18v03) from filename."""
    match = re.search(r"(h\d{2}v\d{2})", filename)
    return match.group(1) if match else None


def extract_date(filename):
    """Extracts date (e.g., A2017001) and converts to datetime."""
    match = re.search(r"\.A(\d{7})\.", filename)
    if match:
        return pd.to_datetime(match.group(1), format="%Y%j")
    return None


def extract_timeseries(year, output_path):
    # 1. Load GeoDataFrame
    print("Loading GeoDataFrame...")
    gdf = gpd.read_file(SAMPLE_POINTS_PATH)
    gdf = gdf[gdf["year"] == year]

    # 2. Reproject to MODIS Sinusoidal
    print(f"Reprojecting {len(gdf)} points to Sinusoidal...")
    gdf_sinu = gdf.to_crs(MODIS_SINU_CRS)

    # Extract X and Y for passing to xarray later
    gdf_sinu["sinu_x"] = gdf_sinu.geometry.x
    gdf_sinu["sinu_y"] = gdf_sinu.geometry.y

    # 3. Catalog Files
    print("Cataloging HDF files...")
    all_files = glob(
        os.path.join(GLASS_LAI_ROOT_DIR, str(year), "**/*.hdf"), recursive=True
    )

    # Group files by Tile ID: {'h18v03': [file1, ...], 'h18v04': [...]}
    files_by_tile = {}
    for f in all_files:
        tid = get_tile_id(os.path.basename(f))
        if tid:
            files_by_tile.setdefault(tid, []).append(f)

    print(f"Found tiles: {list(files_by_tile.keys())}")
    all_results = []

    # 4. Process Each Tile
    for tile_id, file_list in files_by_tile.items():
        print(f"\n--- Processing Tile {tile_id} ---")

        # Sort by date
        file_list.sort(key=lambda x: extract_date(os.path.basename(x)))

        # A. Get Bounding Box of this Tile
        # Open the first file just to read the spatial extent
        try:
            with rioxarray.open_rasterio(file_list[0]) as ref_ds:
                min_x, min_y, max_x, max_y = ref_ds.rio.bounds()
        except Exception as e:
            print(f"Could not read metadata for {tile_id}: {e}")
            continue

        # B. Filter Points: Only keep points inside this tile's extent
        local_points = gdf_sinu[
            (gdf_sinu["sinu_x"] >= min_x)
            & (gdf_sinu["sinu_x"] <= max_x)
            & (gdf_sinu["sinu_y"] >= min_y)
            & (gdf_sinu["sinu_y"] <= max_y)
        ].copy()

        if local_points.empty:
            print(f"No points fall inside {tile_id}. Skipping.")
            continue

        print(f"Found {len(local_points)} points in {tile_id}. Extracting...")

        # C. Load Data Stack & Extract
        try:
            # Create a list to hold the data for each date
            data_arrays = []

            # Loop through every file in this tile group
            for f in tqdm(file_list, desc=f"Loading {tile_id}", leave=False):
                # 1. Open the file using the engine we know works (rioxarray)
                da = rioxarray.open_rasterio(f, chunks="auto")

                # 2. Select Band 0 immediately to simplify dimensions
                lai_layer = da[0].drop_vars("band")

                # 3. Add the time dimension
                dt = extract_date(os.path.basename(f))
                lai_layer = lai_layer.expand_dims(time=[dt])

                # 4. Name it for safety
                lai_layer.name = "LAI"

                data_arrays.append(lai_layer)

            # Concatenate all dates into one time-series block
            ds_stack = xr.concat(data_arrays, dim="time")

            # Define x/y coordinates for vector extraction
            x_coords = xr.DataArray(local_points["sinu_x"].values, dims="point")
            y_coords = xr.DataArray(local_points["sinu_y"].values, dims="point")

            # Extract
            extracted_data = ds_stack.sel(x=x_coords, y=y_coords, method="nearest")

            # Convert to DataFrame
            df_extracted = extracted_data.to_pandas()
            df_extracted.columns = local_points["point_id"].values

            all_results.append(df_extracted)

            # Cleanup
            ds_stack.close()

        except Exception as e:
            print(f"Error extracting {tile_id}: {e}")
            import traceback

            traceback.print_exc()

    # 5. Merge & Export
    if all_results:
        print("\nMerging and saving...")
        final_df = pd.concat(all_results, axis=1)

        # Sort columns to keep IDs orderly
        final_df = final_df.sort_index(axis=1)

        # Apply Scale Factor (if raw data detected)
        # GLASS integer range is usually 0-255 (with scale 0.1)
        if final_df.max().max() > 10:
            print("Applying scale factor (0.1)...")
            final_df = final_df * 0.1

        final_df = final_df.round(1)
        final_df.index.name = "date"
        final_df.to_csv(output_path)
        print(f"Done! Saved to {output_path}")
    else:
        print(
            "No data extracted. Check that your points cover the same area as the downloaded tiles."
        )


# # Extract the LAI timeseries from 2017 to 2021
# for year in range(2017, 2021 + 1):
#     output_path = os.path.join(
#         PROCESSED_DATA_DIR,
#         "crop_type_samples",
#         "LAI",
#         f"crop_type_samples_LAI_{year}.csv",
#     )
#     extract_timeseries(year, output_path)

## Extract SoilGrids data for crop specific LAI points


In [21]:
import ee
import geemap

# Initialize the Earth Engine library
ee.Initialize()

# Load all the ISRIC SoilGrids layers
bdod = ee.Image("projects/soilgrids-isric/bdod_mean").divide(100)
cec = ee.Image("projects/soilgrids-isric/cec_mean").divide(10)
cfvo = ee.Image("projects/soilgrids-isric/cfvo_mean").divide(10)
clay = ee.Image("projects/soilgrids-isric/clay_mean").divide(10)
sand = ee.Image("projects/soilgrids-isric/sand_mean").divide(10)
silt = ee.Image("projects/soilgrids-isric/silt_mean").divide(10)
nitrogen = ee.Image("projects/soilgrids-isric/nitrogen_mean").divide(100)
phh20 = ee.Image("projects/soilgrids-isric/phh2o_mean").divide(10)
soc = ee.Image("projects/soilgrids-isric/soc_mean").divide(100)
ocd = ee.Image("projects/soilgrids-isric/ocd_mean").divide(10)
ocs = ee.Image("projects/soilgrids-isric/ocs_mean").divide(10)

# Combine all layers into a single multi-band image
soil_layers = bdod.addBands(
    [cec, cfvo, clay, sand, silt, nitrogen, phh20, soc, ocd, ocs]
)

In [22]:
soil_layers.bandNames().getInfo()

['bdod_0-5cm_mean',
 'bdod_5-15cm_mean',
 'bdod_15-30cm_mean',
 'bdod_30-60cm_mean',
 'bdod_60-100cm_mean',
 'bdod_100-200cm_mean',
 'cec_0-5cm_mean',
 'cec_5-15cm_mean',
 'cec_15-30cm_mean',
 'cec_30-60cm_mean',
 'cec_60-100cm_mean',
 'cec_100-200cm_mean',
 'cfvo_0-5cm_mean',
 'cfvo_5-15cm_mean',
 'cfvo_15-30cm_mean',
 'cfvo_30-60cm_mean',
 'cfvo_60-100cm_mean',
 'cfvo_100-200cm_mean',
 'clay_0-5cm_mean',
 'clay_5-15cm_mean',
 'clay_15-30cm_mean',
 'clay_30-60cm_mean',
 'clay_60-100cm_mean',
 'clay_100-200cm_mean',
 'sand_0-5cm_mean',
 'sand_5-15cm_mean',
 'sand_15-30cm_mean',
 'sand_30-60cm_mean',
 'sand_60-100cm_mean',
 'sand_100-200cm_mean',
 'silt_0-5cm_mean',
 'silt_5-15cm_mean',
 'silt_15-30cm_mean',
 'silt_30-60cm_mean',
 'silt_60-100cm_mean',
 'silt_100-200cm_mean',
 'nitrogen_0-5cm_mean',
 'nitrogen_5-15cm_mean',
 'nitrogen_15-30cm_mean',
 'nitrogen_30-60cm_mean',
 'nitrogen_60-100cm_mean',
 'nitrogen_100-200cm_mean',
 'phh2o_0-5cm_mean',
 'phh2o_5-15cm_mean',
 'phh2o_15-30cm

In [24]:
col_order = ["point_id"] + soil_layers.bandNames().getInfo()
col_order

['point_id',
 'bdod_0-5cm_mean',
 'bdod_5-15cm_mean',
 'bdod_15-30cm_mean',
 'bdod_30-60cm_mean',
 'bdod_60-100cm_mean',
 'bdod_100-200cm_mean',
 'cec_0-5cm_mean',
 'cec_5-15cm_mean',
 'cec_15-30cm_mean',
 'cec_30-60cm_mean',
 'cec_60-100cm_mean',
 'cec_100-200cm_mean',
 'cfvo_0-5cm_mean',
 'cfvo_5-15cm_mean',
 'cfvo_15-30cm_mean',
 'cfvo_30-60cm_mean',
 'cfvo_60-100cm_mean',
 'cfvo_100-200cm_mean',
 'clay_0-5cm_mean',
 'clay_5-15cm_mean',
 'clay_15-30cm_mean',
 'clay_30-60cm_mean',
 'clay_60-100cm_mean',
 'clay_100-200cm_mean',
 'sand_0-5cm_mean',
 'sand_5-15cm_mean',
 'sand_15-30cm_mean',
 'sand_30-60cm_mean',
 'sand_60-100cm_mean',
 'sand_100-200cm_mean',
 'silt_0-5cm_mean',
 'silt_5-15cm_mean',
 'silt_15-30cm_mean',
 'silt_30-60cm_mean',
 'silt_60-100cm_mean',
 'silt_100-200cm_mean',
 'nitrogen_0-5cm_mean',
 'nitrogen_5-15cm_mean',
 'nitrogen_15-30cm_mean',
 'nitrogen_30-60cm_mean',
 'nitrogen_60-100cm_mean',
 'nitrogen_100-200cm_mean',
 'phh2o_0-5cm_mean',
 'phh2o_5-15cm_mean',
 '

In [ ]:
crop_map_dict = {
    "Wheat": "winter_wheat",
    "Barley": "spring_barley",
    "Maize": "maize",
    "Potatoes": "potato",
    "Sugar Beet": "sugar_beet",
    "Rapeseed": "winter_rapeseed",
}
merged_gdf["crop_type"] = merged_gdf["crop_type"].replace(crop_map_dict)

crops = merged_gdf["crop_type"].unique()

final_df = pd.DataFrame()
for crop in tqdm(crops):
    crop_ee = geemap.gdf_to_ee(
        merged_gdf[merged_gdf["crop_type"] == crop][["point_id", "geometry"]]
    )

    soil_data = soil_layers.reduceRegions(
        collection=crop_ee,
        reducer=ee.Reducer.first(),
        scale=250,
        maxPixelsPerRegion=1e16,
    )

    soil_data = geemap.ee_to_df(soil_data)
    cols = ["point_id"] + [c for c in soil_data.columns if c != "point_id"]
    soil_data = soil_data[cols]
    soil_data.dropna(inplace=True)

    print(f"{crop} | shape: {soil_data.shape}")

    final_df = pd.concat((final_df, soil_data), axis=0, ignore_index=True)

col_order = ["point_id"] + soil_layers.bandNames().getInfo()
final_df = final_df[col_order]
output_path = os.path.join(PROCESSED_DATA_DIR, "soil", "lai", f"LAI_soil.csv")
# final_df.to_csv(output_path, index=False)

  0%|          | 0/6 [00:00<?, ?it/s]

winter_wheat | shape: (1453, 62)
spring_barley | shape: (1572, 62)
maize | shape: (1590, 62)
potato | shape: (1272, 62)
sugar_beet | shape: (1399, 62)
winter_rapeseed | shape: (1401, 62)
